# KG1 V205 - Public Dataset Mining

Objetivo: aproveitar os sinais publicos novos sem treinar as cegas.

Fontes priorizadas:
- Kaggle: `kienngx/nemotron-30b-competition-trainingdata-cot-labels`
- Hugging Face gated: `andy279/nemotron-reasoning-challenge-raw-traces`
- GitHub: `Ayman-Sabek/NVIDIA_Kaggle_Nemotron`
- GitHub: `tonghuikang/nemotron`

Saida: inventario, amostras normalizadas e plano de filtro para `cryptarithm_guess`, `cryptarithm_deduce` e `cipher`.

Este notebook nao treina e nao submete no Kaggle.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    print('Wrote:', path)

def run_cmd(cmd, cwd=None, check=False):
    print('+', ' '.join(map(str, cmd)))
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-5000:])
    if check and p.returncode != 0:
        raise RuntimeError(f'command failed rc={p.returncode}')
    return {'returncode': p.returncode, 'stdout': p.stdout}

IS_COLAB = Path('/content').exists()
WORK_DIR = Path('/content/kg1_v205') if IS_COLAB else Path.cwd() / '.kg1_v205'
OUT_DIR = Path(os.environ.get('KG1_V205_OUT', '/content/drive/MyDrive/KG1_NVIDIA_V205/public_dataset_mining' if IS_COLAB else str(WORK_DIR / 'out')))
DATA_DIR = OUT_DIR / 'data'
for p in [WORK_DIR, OUT_DIR, DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASETS = [
    'kienngx/nemotron-30b-competition-trainingdata-cot-labels',
    'sebmontreal/nvidia-nemotron-model-reasoning-challenge',
]
HF_DATASETS = [
    'andy279/nemotron-reasoning-challenge-raw-traces',
    'andy279/nemotron-reasoning-challenge',
    'jasonkung98/NVIDIA-Nemotron-Model-Reasoning-Challenge',
]
GITHUB_REPOS = [
    'https://github.com/Ayman-Sabek/NVIDIA_Kaggle_Nemotron.git',
    'https://github.com/tonghuikang/nemotron.git',
]

print('OUT_DIR:', OUT_DIR)
print('NO TRAINING. NO KAGGLE SUBMIT.')


In [ ]:
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped/failed:', repr(exc))


## Acao humana obrigatoria se a chave Kaggle foi exposta

Se qualquer chave Kaggle apareceu em notebook, chat, log ou parecer externo, regenere a chave antes de usar esta celula.

Caminho: Kaggle Account -> API -> Expire API Token -> Create New Token.

Depois disponibilize o novo `kaggle.json` em `/content/drive/MyDrive/kaggle.json` ou configure `KAGGLE_USERNAME` e `KAGGLE_KEY` no ambiente.

In [ ]:
def setup_kaggle_credentials():
    has_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    drive_json = Path('/content/drive/MyDrive/kaggle.json')
    if has_env:
        print('Using KAGGLE_USERNAME/KAGGLE_KEY from environment.')
        return True
    if kaggle_json.exists():
        print('Using existing ~/.kaggle/kaggle.json')
        return True
    if drive_json.exists():
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_json, kaggle_json)
        os.chmod(kaggle_json, 0o600)
        print('Copied Kaggle credentials from Drive to ~/.kaggle/kaggle.json')
        return True
    return False

KAGGLE_READY = setup_kaggle_credentials()
print('KAGGLE_READY:', KAGGLE_READY)
if not KAGGLE_READY:
    print('HUMAN ACTION REQUIRED: provide freshly regenerated Kaggle credentials.')


In [ ]:
kaggle_results = []
if KAGGLE_READY:
    try:
        import kaggle  # noqa: F401
    except Exception:
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    for dataset in KAGGLE_DATASETS:
        slug = dataset.replace('/', '__')
        target = DATA_DIR / 'kaggle' / slug
        target.mkdir(parents=True, exist_ok=True)
        result = {'dataset': dataset, 'target': str(target)}
        rc = run_cmd(['kaggle', 'datasets', 'download', '-d', dataset, '-p', str(target), '--unzip'])
        result['download_returncode'] = rc['returncode']
        files = []
        for path in target.rglob('*'):
            if path.is_file():
                rel = path.relative_to(target).as_posix()
                files.append({'path': rel, 'bytes': path.stat().st_size, 'sha256': sha256_file(path) if path.stat().st_size < 200_000_000 else None})
        result['files'] = files[:200]
        result['file_count'] = len(files)
        kaggle_results.append(result)
else:
    kaggle_results.append({'status': 'blocked_human_action_required', 'reason': 'missing fresh Kaggle credentials'})

write_json(OUT_DIR / 'V205_kaggle_dataset_downloads.json', kaggle_results)


In [ ]:
hf_results = []
hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
try:
    import datasets  # noqa: F401
except Exception:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
from datasets import load_dataset

for ds_name in HF_DATASETS:
    row = {'dataset': ds_name}
    try:
        ds = load_dataset(ds_name, split='train', streaming=True, token=hf_token)
        sample = []
        for i, item in enumerate(ds):
            sample.append({k: item.get(k) for k in list(item.keys())[:40]})
            if i >= 19:
                break
        sample_path = DATA_DIR / 'hf_samples' / (ds_name.replace('/', '__') + '_sample.json')
        write_json(sample_path, sample)
        row.update({'ok': True, 'sample_path': str(sample_path), 'sample_count': len(sample), 'columns': list(sample[0].keys()) if sample else []})
    except Exception as exc:
        row.update({'ok': False, 'error': repr(exc)})
        if 'andy279' in ds_name and not hf_token:
            row['human_action'] = 'Set HF_TOKEN after accepting dataset terms on Hugging Face.'
    hf_results.append(row)

write_json(OUT_DIR / 'V205_hf_dataset_samples.json', hf_results)


In [ ]:
repo_results = []
repos_dir = DATA_DIR / 'github_repos'
repos_dir.mkdir(parents=True, exist_ok=True)
for url in GITHUB_REPOS:
    name = url.rstrip('/').split('/')[-1].replace('.git', '')
    target = repos_dir / name
    if target.exists():
        rc = run_cmd(['git', '-C', str(target), 'pull', '--ff-only'])
    else:
        rc = run_cmd(['git', 'clone', '--depth', '1', url, str(target)])
    files = []
    if target.exists():
        for path in target.rglob('*'):
            if path.is_file() and '.git' not in path.parts:
                rel = path.relative_to(target).as_posix()
                if any(rel.lower().endswith(ext) for ext in ['.py', '.ipynb', '.md', '.json', '.csv', '.yaml', '.yml', '.txt']):
                    files.append({'path': rel, 'bytes': path.stat().st_size})
    repo_results.append({'url': url, 'target': str(target), 'returncode': rc['returncode'], 'files': files[:300], 'file_count': len(files)})

write_json(OUT_DIR / 'V205_github_repo_inventory.json', repo_results)


In [ ]:
KEYWORDS = ['cryptarithm', 'cipher', 'equation', 'guess', 'deduce', 'verify', 'reasoner', 'solver', 'answer', 'boxed']
interesting = []
for root in [DATA_DIR / 'kaggle', DATA_DIR / 'github_repos']:
    if not root.exists():
        continue
    for path in root.rglob('*'):
        if not path.is_file() or path.stat().st_size > 20_000_000:
            continue
        rel = path.relative_to(DATA_DIR).as_posix()
        low = rel.lower()
        score = sum(1 for k in KEYWORDS if k in low)
        if score == 0 and path.suffix.lower() not in ['.csv', '.json', '.jsonl', '.parquet', '.py', '.ipynb']:
            continue
        interesting.append({'path': rel, 'bytes': path.stat().st_size, 'keyword_score': score, 'suffix': path.suffix.lower()})

interesting.sort(key=lambda r: (-r['keyword_score'], r['bytes'], r['path']))
write_json(OUT_DIR / 'V205_interesting_files.json', interesting[:500])
print(json.dumps(interesting[:80], indent=2))


In [ ]:
decision = {
    'generated_at': utc_now(),
    'decision': 'V205_DATASET_MINING_READY_OR_BLOCKED',
    'no_training_executed': True,
    'no_kaggle_submit_executed': True,
    'kaggle_ready': bool(KAGGLE_READY),
    'hf_token_present': bool(hf_token),
    'artifacts': {
        'kaggle_downloads': str(OUT_DIR / 'V205_kaggle_dataset_downloads.json'),
        'hf_samples': str(OUT_DIR / 'V205_hf_dataset_samples.json'),
        'github_inventory': str(OUT_DIR / 'V205_github_repo_inventory.json'),
        'interesting_files': str(OUT_DIR / 'V205_interesting_files.json'),
    },
    'next_filter': {
        'priority_categories': ['cryptarithm_guess', 'cryptarithm_deduce', 'cipher', 'equation_numeric_guess'],
        'allow_into_training_only_if': [
            'answer is verified by local official verifier or deterministic solver',
            'category is known',
            'no id-specific override',
            'format passes boxed/final answer parser',
            'sample improves a known V194 weakness without adding official360 regression risk',
        ],
    },
}
blocked = []
if not KAGGLE_READY:
    blocked.append('fresh Kaggle credentials required for Kaggle datasets')
if not hf_token:
    blocked.append('HF_TOKEN required for gated andy279 datasets after accepting terms')
decision['blocked_human_actions'] = blocked
write_json(OUT_DIR / 'V205_FINAL_DECISION.json', decision)
print(json.dumps(decision, indent=2))
if blocked:
    raise RuntimeError('HUMAN ACTION REQUIRED: ' + '; '.join(blocked))
